# 밀도범함수이론 실습

**Density Functional Theory · DFT**

전자 밀도를 중심으로 물질의 전자 구조와 에너지를 계산하는 양자역학적 방법.

소재 분야에서 이해하기: 결정 후보의 에너지를 계산해 안정성을 비교한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Materials Project 용어집](https://docs.materialsproject.org/frequently-asked-questions/glossary-of-terms)

## 1. 전자 구조 계산의 골격

DFT 자체를 여기서 돌릴 수는 없지만, "포텐셜에서 전자 상태를 구하고 밀도를 얻는" 골격은
1차원 문제로 직접 풀 수 있습니다. 유한차분으로 슈뢰딩거 방정식의 고유상태를 구합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from scipy.linalg import eigh

grid_points = 400
length = 1.0
x = np.linspace(0, length, grid_points)
h = x[1] - x[0]

def solve(potential, states=4):
    """(-1/2) d^2/dx^2 + V 의 고유값 문제 (원자단위)."""
    kinetic = (np.diag(np.full(grid_points - 2, 1.0)) * -2
               + np.diag(np.ones(grid_points - 3), 1) + np.diag(np.ones(grid_points - 3), -1))
    hamiltonian = -0.5 * kinetic / h ** 2 + np.diag(potential[1:-1])
    values, vectors = eigh(hamiltonian)
    wavefunctions = np.zeros((grid_points, states))
    for index in range(states):
        wavefunctions[1:-1, index] = vectors[:, index]
        wavefunctions[:, index] /= np.sqrt(np.trapezoid(wavefunctions[:, index] ** 2, x))
    return values[:states], wavefunctions

energies, wavefunctions = solve(np.zeros(grid_points))
print('무한 우물 고유에너지 (계산 / 해석해 n^2 pi^2 / 2L^2):')
for n in range(1, 5):
    exact = n ** 2 * np.pi ** 2 / (2 * length ** 2)
    print('  n=%d  %.3f / %.3f' % (n, energies[n - 1], exact))

## 2. 포텐셜을 바꾸면 상태가 바뀝니다

In [ ]:
barrier = np.zeros(grid_points)
barrier[(x > 0.45) & (x < 0.55)] = 300.0        # 가운데 장벽 (두 우물처럼 만듭니다)
energies_barrier, wavefunctions_barrier = solve(barrier)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for axis, (title, values, functions, potential) in zip(axes, [
        ('single well', energies, wavefunctions, np.zeros(grid_points)),
        ('barrier in the middle', energies_barrier, wavefunctions_barrier, barrier)]):
    for index in range(3):
        axis.plot(x, functions[:, index] * 0.02 + values[index], label='n=%d' % (index + 1))
    axis.plot(x, np.clip(potential, 0, 120), 'k--', lw=1)
    axis.set_title(title); axis.set_xlabel('x'); axis.set_ylabel('energy'); axis.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('장벽이 있으면 1,2번 상태의 에너지가 거의 같아집니다(축퇴): %.3f, %.3f'
      % (energies_barrier[0], energies_barrier[1]))

## 3. 전자 밀도와 자체일관성

DFT는 밀도로부터 포텐셜을 만들고, 그 포텐셜로 다시 밀도를 구하는 것을 수렴할 때까지 반복합니다.
아주 단순한 형태로 그 루프를 돌려봅니다.

In [ ]:
def density_from_states(functions, electrons=2):
    return sum(functions[:, index] ** 2 for index in range(electrons))

potential = np.zeros(grid_points)
history = []
for iteration in range(12):
    values, functions = solve(potential)
    density = density_from_states(functions)
    # 아주 단순한 반발 포텐셜 (실제 DFT의 교환상관 범함수를 대신한 장난감 항)
    new_potential = 8.0 * density
    potential = 0.4 * new_potential + 0.6 * potential      # 혼합으로 수렴 안정화
    history.append(values[0])
    print('반복 %2d  최저 에너지 %.4f' % (iteration + 1, values[0]))

plt.plot(history, 'o-'); plt.xlabel('iteration'); plt.ylabel('lowest eigenvalue'); plt.show()
print('\n실제 DFT도 이처럼 자체일관 루프를 돕니다. 결과는 쓰는 범함수와 수렴 조건에 의존하므로,')
print('데이터로 쓸 때는 계산 조건을 반드시 함께 기록해야 합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#dft)을 여세요.